In [1]:
from NoisyCircuits import QuantumCircuit as QC
from NoisyCircuits.utils.CreateNoiseModel import GetNoiseModel, CreateNoiseModel
import pickle
import os
import json
import numpy as np

2026-03-10 13:15:26,699	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


In [2]:
api_json = json.load(open(os.path.join(os.path.expanduser("~"), "ibm_api.json"), "r"))
token = api_json["apikey"] # Replace with your Token
service_crn = api_json["service-crn"] # Replace with your Service CRN
backend_name = "ibm_fez"
num_qubits = 4
num_cores = 75
num_trajectories = 20
threshold = 1e-4
jsonize = True
verbose = False
# Options: "heron", "eagle". Note that "eagle" is now deprecated and only available in simulation mode (i.e., no noise model from hardware).
qpu_type = "heron" 

In [3]:
if qpu_type == "eagle":
    file_path = "https://raw.githubusercontent.com/Sats2/NoisyCircuits/main/noise_models/Noise_Model_Eagle_QPU.pkl"
    noise_model = pickle.load(open(file_path, "rb"))
elif qpu_type == "heron":
    file_path = "https://raw.githubusercontent.com/Sats2/NoisyCircuits/CustomNoiseModel/noise_models/Sample_Noise_Model_Heron_QPU.csv"
    noise_model = CreateNoiseModel(calibration_data_file=file_path, 
                                   basis_gates=[["x", "sx", "rz", "rx"], ["cz", "rzz"]]).create_noise_model()
else:
    raise ValueError("Invalid qpu_type. Choose either 'heron' or 'eagle'.")

In [4]:
circuit_qiskit = QC(num_qubits=num_qubits, 
                    noise_model=noise_model, 
                    num_cores=num_cores,
                    backend_qpu_type=qpu_type, 
                    num_trajectories=num_trajectories, 
                    sim_backend="qiskit",
                    threshold=threshold, 
                    jsonize=jsonize,
                    verbose=verbose)
circuit_pennylane = QC(num_qubits=num_qubits, 
                    noise_model=noise_model, 
                    num_cores=num_cores,
                    backend_qpu_type=qpu_type, 
                    num_trajectories=num_trajectories, 
                    sim_backend="pennylane",
                    threshold=threshold, 
                    jsonize=jsonize,
                    verbose=verbose)
circuit_qulacs = QC(num_qubits=num_qubits, 
                    noise_model=noise_model, 
                    num_cores=num_cores,
                    backend_qpu_type=qpu_type, 
                    num_trajectories=num_trajectories, 
                    sim_backend="qulacs",
                    threshold=threshold, 
                    jsonize=jsonize,
                    verbose=verbose)

Successfully switched backend to qiskit.


2026-03-10 13:15:57,449	INFO worker.py:2007 -- Started a local Ray instance.
/Users/adam-ukj7r05xnu2fywx/miniconda3/envs/NoisyCircuits/lib/python3.10/site-packages/ray/_private/worker.py:2046: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


Successfully switched backend to pennylane.


2026-03-10 13:16:03,063	INFO worker.py:1839 -- Calling ray.init() again after it has already been called.


Successfully switched backend to qulacs.


2026-03-10 13:16:07,193	INFO worker.py:1839 -- Calling ray.init() again after it has already been called.


In [5]:
def random_circuit(circuit_list, num_qubits, depth):
    for circuit in circuit_list:
        circuit.refresh()
    single_qubit_gates = {
        "x": lambda circuit, q: circuit.X(q[0]),
        "h": lambda circuit, q: circuit.H(q[0]),
        "sx": lambda circuit, q: circuit.SX(q[0]),
        "rx": lambda circuit, q: circuit.RX(q[1], q[0]),
        "ry": lambda circuit, q: circuit.RY(q[1], q[0]),
        "y": lambda circuit, q: circuit.Y(q[0]),
        "z": lambda circuit, q: circuit.Z(q[0])
    }
    two_qubit_gates = {
        "cz": lambda circuit, q: circuit.CZ(q[0], q[1]),
        "crx": lambda circuit, q: circuit.CRX(q[2], q[0], q[1]),
        "crz": lambda circuit, q: circuit.CRZ(q[2], q[0], q[1]),
        "cry": lambda circuit, q: circuit.CRY(q[2], q[0], q[1])
    }
    for i in range(num_qubits):
        for circuit in circuit_list:
            circuit.H(i)
    for _ in range(depth):
        for qubit in range(num_qubits):
            gate_choice = np.random.choice(list(single_qubit_gates.keys()))
            params = np.random.uniform(-2*np.pi, 2*np.pi)
            for circuit in circuit_list:
                single_qubit_gates[gate_choice](circuit, [qubit, params])
        for qubit in range(num_qubits - 1):
            gate_choice = np.random.choice(list(two_qubit_gates.keys()))
            params = np.random.uniform(-2*np.pi, 2*np.pi)
            for circuit in circuit_list:
                two_qubit_gates[gate_choice](circuit, [qubit, qubit + 1, params])

In [6]:
def fidelity(p1, p2):
    fid = np.sum(np.sqrt(p1 * p2))
    return fid

In [7]:
random_circuit([circuit_qiskit, circuit_pennylane, circuit_qulacs], num_qubits=num_qubits, depth=200)

In [8]:
p_qiskit_pure = circuit_qiskit.run_pure_state(list(range(num_qubits)))
p_pennylane_pure = circuit_pennylane.run_pure_state(list(range(num_qubits)))
p_qulacs_pure = circuit_qulacs.run_pure_state(list(range(num_qubits)))
print("Fidelity between Qiskit and Pennylane: ", fidelity(p_qiskit_pure, p_pennylane_pure))
print("Fidelity between Qiskit and Qulacs: ", fidelity(p_qiskit_pure, p_qulacs_pure))
print("Fidelity between Pennylane and Qulacs: ", fidelity(p_pennylane_pure, p_qulacs_pure))

Fidelity between Qiskit and Pennylane:  1.0000000000000013
Fidelity between Qiskit and Qulacs:  1.0000000000000027
Fidelity between Pennylane and Qulacs:  1.0000000000000013


In [ ]:
p_qiskit_dm = circuit_qiskit.run_with_density_matrix(list(range(num_qubits)))
p_pennylane_dm = circuit_pennylane.run_with_density_matrix(list(range(num_qubits)))
p_qulacs_dm = circuit_qulacs.run_with_density_matrix(list(range(num_qubits)))
print("Fidelity between Qiskit and Pennylane: ", fidelity(p_qiskit_dm, p_pennylane_dm))
print("Fidelity between Qiskit and Qulacs: ", fidelity(p_qiskit_dm, p_qulacs_dm))
print("Fidelity between Pennylane and Qulacs: ", fidelity(p_pennylane_dm, p_qulacs_dm))

Fidelity between Qiskit and Pennylane:  0.9999999999998028
Fidelity between Qiskit and Qulacs:  0.9999999999997711
Fidelity between Pennylane and Qulacs:  0.9999999999998761


[2026-03-10 13:16:28,164 E 1003621 1004240] core_worker_process.cc:842: Failed to establish connection to the metrics exporter agent. Metrics will not be exported. Exporter agent status: RpcError: Running out of retries to initialize the metrics agent. rpc_code: 14


In [10]:
print("Fidelity between Qiskit pure state and density matrix: ", fidelity(p_qiskit_pure, p_qiskit_dm))
print("Fidelity between Pennylane pure state and density matrix: ", fidelity(p_pennylane_pure, p_pennylane_dm))
print("Fidelity between Qulacs pure state and density matrix: ", fidelity(p_qulacs_pure, p_qulacs_dm))

Fidelity between Qiskit pure state and density matrix:  0.9368284987518565
Fidelity between Pennylane pure state and density matrix:  0.9368284987519508
Fidelity between Qulacs pure state and density matrix:  0.9368284987519235


In [11]:
import time

In [12]:
t0 = time.perf_counter_ns()
p_qiskit = circuit_qiskit.execute(list(range(num_qubits)), 1000)
t1 = time.perf_counter_ns()
print(f"Execution time for Qiskit: {(t1 - t0)*1e-9} seconds.")
t0 = time.perf_counter_ns()
p_pennylane = circuit_pennylane.execute(list(range(num_qubits)), 1000)
t1 = time.perf_counter_ns()
print(f"Execution time for Pennylane: {(t1 - t0)*1e-9} seconds.")
t0 = time.perf_counter_ns()
p_qulacs = circuit_qulacs.execute(list(range(num_qubits)), 1000)
t1 = time.perf_counter_ns()
print(f"Execution time for Qulacs: {(t1 - t0)*1e-9} seconds.")

Execution time for Qiskit: 100.51229388600001 seconds.
Execution time for Pennylane: 113.055789211 seconds.
Execution time for Qulacs: 15.299513631000002 seconds.


In [13]:
print("Fidelity between Qiskit and Pennylane: ", fidelity(p_qiskit, p_pennylane))
print("Fidelity between Qiskit and Qulacs: ", fidelity(p_qiskit, p_qulacs))
print("Fidelity between Pennylane and Qulacs: ", fidelity(p_pennylane, p_qulacs))

Fidelity between Qiskit and Pennylane:  0.9999999999999329
Fidelity between Qiskit and Qulacs:  1.0000000000000004
Fidelity between Pennylane and Qulacs:  0.9999999999999329


In [14]:
for circuit in [circuit_qiskit, circuit_pennylane, circuit_qulacs]:
    circuit.shutdown()